In [28]:
# net liquidity central bank cleaner

import pandas as pd
import holidays
from datetime import timedelta

def calculate_rbi_release_dates(file_path, output_path):
    # 1. Load the CSV (skipping metadata rows if necessary)
    # Adjust 'skiprows' depending on exactly how many header rows are in your raw export
    df = pd.read_csv(file_path, skiprows=6) 
    
    # Clean column names (strip whitespace)
    df.columns = df.columns.str.strip()
    
    # Identify your date column. Assuming it's named 'Date' or similar from the DBIE
    # Convert it to pandas datetime
    df['Operation_Date'] = pd.to_datetime(df['Date'], errors='coerce')
    
    # Drop rows where date parsing failed (like totals or footers)
    df = df.dropna(subset=['Operation_Date']).copy()
    
    # 2. Initialize Indian National/Market Holidays
    # RBI follows major national holidays. We can extend this list if needed.
    tg_holidays = holidays.India(years=list(range(df['Operation_Date'].dt.year.min(), df['Operation_Date'].dt.year.max() + 1)))

    def get_actual_release_date(op_date):
        target_release = op_date + timedelta(days=1)
        # Skip weekends
        if target_release.weekday() == 5:  # Saturday
            target_release = target_release + timedelta(days=2)  # Move to Monday
        elif target_release.weekday() == 6:  # Sunday
            target_release = target_release + timedelta(days=1)  # Move to Monday
    
        # Then check if that day is a holiday
        if target_release in tg_holidays:
            target_release = target_release + timedelta(days=1)
            # Check again in case the next day is also a holiday or weekend
            if target_release.weekday() == 6:
                target_release = target_release + timedelta(days=1)
            if target_release in tg_holidays:
                target_release = target_release + timedelta(days=1)
        return target_release

    # 3. Apply the logic to generate the release date column
    df['Release_Date'] = df['Operation_Date'].apply(get_actual_release_date)
    
    # Format dates back to clean strings for readability if preferred
    df['Operation_Date'] = df['Operation_Date'].dt.strftime('%Y-%m-%d')
    df['Release_Date'] = df['Release_Date'].dt.strftime('%Y-%m-%d')
    
    # Move 'Release_Date' to the front for easy backtesting alignment
    cols = ['Operation_Date', 'Release_Date'] + [col for col in df.columns if col not in ['Date', 'Operation_Date', 'Release_Date']]
    df = df[cols]
    
    # 4. Save the modified file
    df.to_csv(output_path, index=False)
    print(success_msg := f"File successfully processed! Saved to: {output_path}")

# Run the function
calculate_rbi_release_dates('RBI_Liquidity.csv', 'RBI_Liquidity_With_Lags.csv')

File successfully processed! Saved to: RBI_Liquidity_With_Lags.csv


In [29]:
# WACR repo spread cleaner

import pandas as pd
import holidays
from datetime import timedelta

def process_wacr_wss_file(file_path, output_path):
    # 1. Load your WACR file
    df = pd.read_csv(file_path, skiprows=4)
    
    # Clean structural artifacts (like fiscal year labels '2026-27' printed as rows)
    df.columns = ['Date_Raw', 'WACR', 'WACR_Duplicate', 'Empty1', 'Empty2']
    df['Date_Clean'] = df['Date_Raw'].str.strip()
    df = df[df['Date_Clean'].str.contains('-', na=False)] # Must look like a date string
    
    # Convert and filter out non-numeric values
    df['Operation_Date'] = pd.to_datetime(df['Date_Clean'], errors='coerce', format='%d-%b-%y')
    df['WACR'] = pd.to_numeric(df['WACR'], errors='coerce')
    df = df.dropna(subset=['Operation_Date', 'WACR']).copy()
    
    # 2. AUTOMATED REPO RATE FETCHING (Via Wikipedia MPC Table Tracker)
    try:
        url = "https://en.wikipedia.org/wiki/Monetary_Policy_Committee_of_India"
        tables = pd.read_html(url)
        
        # Locate the table tracking the policy actions historically
        # We find the table that mentions 'Repo Rate' or historical cuts
        mpc_table = None
        for t in tables:
            if 'Repo Rate' in t.columns.get_level_values(0) or 'Repo rate' in ''.join(t.columns.astype(str)):
                mpc_table = t
                break
                
        # Clean the fetched table to build a dynamic lookup timeline
        mpc_table.columns = [str(c).strip() for c in mpc_table.columns]
        # Standardize column headers based on Wiki schema ('Date' and 'Repo Rate')
        date_col = [c for c in mpc_table.columns if 'Date' in c][0]
        repo_col = [c for c in mpc_table.columns if 'Repo' in c][0]
        
        repo_history = mpc_table[[date_col, repo_col]].copy()
        repo_history['Effective_Date'] = pd.to_datetime(repo_history[date_col], errors='coerce')
        repo_history['Rate'] = pd.to_numeric(repo_history[repo_col].astype(str).str.extract(r'(\d+\.\d+)')[0], errors='coerce')
        repo_history = repo_history.dropna().sort_values(by='Effective_Date')
        
    except Exception as e:
        print(f"Web-fetch failed ({e}). Falling back to a structured runtime dynamic index...")
        # Precise dynamic index mapping up to mid-2026 if running completely offline:
        repo_timeline = {
            '2025-12-05': 5.25, '2025-06-06': 5.50, '2025-04-09': 6.00, '2025-02-07': 6.25,
            '2023-02-08': 6.50, '2022-12-07': 6.25, '2022-09-30': 5.90, '2022-08-05': 5.40,
            '2022-06-08': 4.90, '2022-05-04': 4.40, '2020-05-22': 4.00, '2020-03-27': 4.40,
            '2019-10-04': 5.15, '2019-08-07': 5.40, '2019-06-06': 5.75, '2019-04-04': 6.00,
            '2019-02-07': 6.25, '2018-08-01': 6.50, '2018-06-06': 6.25, '2017-08-02': 6.00,
            '2016-10-04': 6.25, '2016-04-05': 6.50, '2015-09-29': 6.75, '2015-06-02': 7.25
        }
        repo_history = pd.DataFrame(list(repo_timeline.items()), columns=['Effective_Date', 'Rate'])
        repo_history['Effective_Date'] = pd.to_datetime(repo_history['Effective_Date'])
        repo_history = repo_history.sort_values(by='Effective_Date')

    # Vectorized point-in-time lookup to find applicable repo rate for every row's date
    df = df.sort_values(by='Operation_Date')
    df = pd.merge_asof(df, repo_history[['Effective_Date', 'Rate']], 
                       left_on='Operation_Date', right_on='Effective_Date', 
                       direction='backward')
    df.rename(columns={'Rate': 'Repo_Rate'}, inplace=True)
    df['Repo_Rate'] = df['Repo_Rate'].fillna(6.00) # Fallback baseline value for earliest 2005 points

    # 3. Dynamic Holiday-Adjusted Release Dates
    start_year, end_year = df['Operation_Date'].dt.year.min(), df['Operation_Date'].dt.year.max()
    tg_holidays = holidays.India(years=list(range(int(start_year), int(end_year) + 1)))

    def calculate_strict_wss_release(op_date):
        days_until_next_friday = (4 - op_date.weekday()) + 7
        target_release_friday = op_date + timedelta(days=days_until_next_friday)
        
        if target_release_friday in tg_holidays:
            actual_release = target_release_friday - timedelta(days=1)
            if actual_release in tg_holidays:
                actual_release = actual_release - timedelta(days=1)
            return actual_release
        return target_release_friday

    df['Release_Date'] = df['Operation_Date'].apply(calculate_strict_wss_release)

    # 4. Calculate Final Spread Metric
    df['WACR_Repo_Spread'] = df['WACR'] - df['Repo_Rate']

    # Final Formatting
    df['Operation_Date'] = df['Operation_Date'].dt.strftime('%Y-%m-%d')
    df['Release_Date'] = df['Release_Date'].dt.strftime('%Y-%m-%d')
    
    final_cols = ['Operation_Date', 'Release_Date', 'WACR', 'Repo_Rate', 'WACR_Repo_Spread']
    final_df = df[final_cols].sort_values(by='Operation_Date', ascending=False)

    # Save output CSV
    final_df.to_csv(output_path, index=False)
    print(f"File successfully created: {output_path}")

# Run pipeline
process_wacr_wss_file('Daily Weighted Average Call_Notice Money Rates.csv', 'WACR_Spread_Automated.csv')

Web-fetch failed (<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1082)>). Falling back to a structured runtime dynamic index...
File successfully created: WACR_Spread_Automated.csv


In [30]:
# 91-Day T-Bill Yield repo spread cleaning

import pandas as pd
import holidays
from datetime import timedelta

def process_tbill_91day_auctions(file_path, output_path):
    # 1. Load the CSV (Skipping the raw structural header metadata blocks)
    df = pd.read_csv(file_path, skiprows=4)
    
    # Map the relevant columns dynamically based on the RBI layout grid
    # Column 0 = Date of Auction, Column 12 = Implicit Yield at Cut-off Price
    df.columns = [
        'Auction_Date_Raw', 'Issue_Date', 'Notified_Amt', 'Bids_Recv_Num', 
        'Bids_Recv_Comp', 'Bids_Recv_NonComp', 'Bids_Acc_Num', 'Bids_Acc_Comp', 
        'Bids_Acc_NonComp', 'Devolvement', 'Total_Issue', 'Cutoff_Price', 
        'Implicit_Yield', 'Outstanding_Amt', 'Wtd_Avg_Price', 'Wtd_Avg_Yield'
    ]
    
    # Clean structural row noise (like fiscal year headers '2026-27' interspersed in rows)
    df['Auction_Date_Clean'] = df['Auction_Date_Raw'].str.strip()
    df = df[df['Auction_Date_Clean'].str.contains('-', na=False)] 
    
    # Safe parse the Auction Date and target Yield index
    df['Auction_Date'] = pd.to_datetime(df['Auction_Date_Clean'], errors='coerce', format='%d-%b-%Y')
    df['Implicit_Yield'] = pd.to_numeric(df['Implicit_Yield'], errors='coerce')
    df = df.dropna(subset=['Auction_Date', 'Implicit_Yield']).copy()

    # 2. Implement the 2-Day Release Lag (Wednesday Auction -> Friday WSS Release)
    start_year = df['Auction_Date'].dt.year.min()
    end_year = df['Auction_Date'].dt.year.max()
    tg_holidays = holidays.India(years=list(range(int(start_year), int(end_year) + 1)))

    def calculate_tb_release_date(auc_date):
        # Base rule: Auction + 2 days shifts a Wednesday to Friday
        target_release = auc_date + timedelta(days=2)
        
        # Holiday Adjustments: If Friday WSS release day is closed, RBI pushes on Thursday
        if target_release in tg_holidays:
            actual_release = target_release - timedelta(days=1)
            if actual_release in tg_holidays:
                actual_release = actual_release - timedelta(days=1)
            return actual_release
        return target_release

    df['Release_Date'] = df['Auction_Date'].apply(calculate_tb_release_date)

    # 3. Vectorized Historical Point-in-Time Repo Rate Mapping
    repo_timeline = {
        '2025-12-05': 5.25, '2025-06-06': 5.50, '2025-04-09': 6.00, '2025-02-07': 6.25,
        '2023-02-08': 6.50, '2022-12-07': 6.25, '2022-09-30': 5.90, '2022-08-05': 5.40,
        '2022-06-08': 4.90, '2022-05-04': 4.40, '2020-05-22': 4.00, '2020-03-27': 4.40,
        '2019-10-04': 5.15, '2019-08-07': 5.40, '2019-06-06': 5.75, '2019-04-04': 6.00,
        '2019-02-07': 6.25, '2018-08-01': 6.50, '2018-06-06': 6.25, '2017-08-02': 6.00,
        '2016-10-04': 6.25, '2016-04-05': 6.50, '2015-09-29': 6.75, '2015-06-02': 7.25,
        '2015-03-04': 7.50, '2015-01-15': 7.75, '2014-01-28': 8.00, '2013-10-29': 7.75,
        '2013-09-20': 7.50, '2013-05-03': 7.25, '2013-03-19': 7.50, '2013-01-29': 7.75,
        '2012-04-17': 8.00, '2011-10-25': 8.50, '2011-09-16': 8.25, '2011-07-26': 8.00,
        '2011-06-16': 7.50, '2011-05-03': 7.25, '2011-03-17': 6.75, '2011-01-25': 6.50,
        '2010-11-02': 6.25, '2010-09-16': 6.00, '2010-07-27': 5.75, '2010-07-02': 5.50,
        '2010-04-20': 5.25, '2010-03-19': 5.00, '2009-04-21': 4.75, '2009-03-04': 5.00,
        '2009-01-02': 5.50, '2008-12-06': 6.50, '2008-11-01': 7.50, '2008-10-20': 8.00,
        '2008-07-29': 9.00, '2008-06-24': 8.75, '2008-06-11': 8.00, '2007-03-30': 7.75,
        '2007-01-31': 7.50, '2006-10-31': 7.25, '2006-07-25': 7.00, '2006-06-09': 6.75,
        '2006-01-24': 6.50, '2005-10-25': 6.25
    }
    repo_df = pd.DataFrame(list(repo_timeline.items()), columns=['Effective_Date', 'Repo_Rate'])
    repo_df['Effective_Date'] = pd.to_datetime(repo_df['Effective_Date'])
    repo_df = repo_df.sort_values(by='Effective_Date')

    # Point-in-time merge matching back to the exact Auction Day conditions
    df = df.sort_values(by='Auction_Date')
    df = pd.merge_asof(df, repo_df, left_on='Auction_Date', right_on='Effective_Date', direction='backward')
    df['Repo_Rate'] = df['Repo_Rate'].fillna(6.00) # Baseline global fallback for early 2000s bounds

    # 4. Compute T-Bill Yield vs Policy Repo Rate Spread
    df['TBill_Repo_Spread'] = df['Implicit_Yield'] - df['Repo_Rate']

    # 5. Column Pruning and Serialization Formatting
    df['Auction_Date'] = df['Auction_Date'].dt.strftime('%Y-%m-%d')
    df['Release_Date'] = df['Release_Date'].dt.strftime('%Y-%m-%d')
    
    output_cols = ['Auction_Date', 'Release_Date', 'Implicit_Yield', 'Repo_Rate', 'TBill_Repo_Spread']
    final_df = df[output_cols].sort_values(by='Auction_Date', ascending=False)

    # Export clean CSV
    final_df.to_csv(output_path, index=False)
    print(f"File processed and compiled at: {output_path}")

# Run function on your T-Bill dataset
process_tbill_91day_auctions('Auctions of 91-Day Government of India Treasury Bills.csv', 'TBill_91D_Spread_Clean.csv')

File processed and compiled at: TBill_91D_Spread_Clean.csv


/var/folders/q5/pq31bcf10fv9ktv6b14g42zr0000gn/T/ipykernel_98103/2174349579.py:32: UserWarning: Requested Holidays are available only from 2001 to 2035.
  tg_holidays = holidays.India(years=list(range(int(start_year), int(end_year) + 1)))


In [31]:
#C/D differential cleaner

import pandas as pd
from datetime import timedelta

def process_banking_aggregates(file_path, output_path):
    # 1. Load data safely (Skipping 5 rows gets us to the true header line)
    df = pd.read_csv(file_path, skiprows=5)
    
    # Drop the first row containing the column index numbers (1, 2, 3...)
    df = df.iloc[1:].copy()
    
    # Clean the column names of trailing whitespace
    df.columns = [str(c).strip() for c in df.columns]

    # Map the relevant columns dynamically
    df.rename(columns={
        'Fortnight ended': 'Period_Date_Raw',
        'Aggregate deposits (2+3)': 'Aggregate_Deposits_Raw',
        'Bank Credit (11+12)': 'Bank_Credit_Raw'
    }, inplace=True)

    # 2. Parse Period Date
    df['Period_Date'] = pd.to_datetime(df['Period_Date_Raw'], errors='coerce')
    df = df.dropna(subset=['Period_Date']).copy()

    # Clean numeric strings (remove commas) and convert to float
    df['Aggregate_Deposits'] = pd.to_numeric(df['Aggregate_Deposits_Raw'].astype(str).str.replace(',', ''), errors='coerce')
    df['Bank_Credit'] = pd.to_numeric(df['Bank_Credit_Raw'].astype(str).str.replace(',', ''), errors='coerce')

    # 3. Calculate 2nd Column: "Following Week" Friday Release Date
    def get_release_date(date):
        # Python weekdays: Monday = 0 ... Friday = 4
        # Calculate days to the Friday of the *current* calendar week, then add 7 days to force it into next week
        days_to_next_friday = (4 - date.weekday()) + 7
        return date + timedelta(days=days_to_next_friday)

    df['Release_Date'] = df['Period_Date'].apply(get_release_date)

    # Sort ascending chronologically to ensure accurate historical point-in-time calculation
    df = df.sort_values('Period_Date')

    # 4. YoY Growth Calculation Engine (See explanation below)
    df_1yr_ago = df[['Period_Date', 'Aggregate_Deposits', 'Bank_Credit']].copy()
    
    # Create a target lookup date shifted back exactly 1 calendar year
    df_1yr_ago['Target_Date'] = df_1yr_ago['Period_Date'] + pd.DateOffset(years=1)

    # Use merge_asof to find the historical row closest to the Target_Date
    df = pd.merge_asof(
        df, 
        df_1yr_ago.rename(columns={
            'Aggregate_Deposits': 'Agg_Dep_1YrAgo', 
            'Bank_Credit': 'Bank_Cred_1YrAgo', 
            'Period_Date': 'Matched_Historical_Date'
        }),
        left_on='Period_Date',
        right_on='Target_Date',
        direction='nearest',
        tolerance=pd.Timedelta(days=15) # Prevents mapping if data is wildly missing
    )

    # Calculate differential
    df['C/D_differential'] = df['Bank_Credit'] / df['Aggregate_Deposits']
    

    # Final Formatting: Prune columns and sort descending
    df['Period_Date'] = df['Period_Date'].dt.strftime('%Y-%m-%d')
    df['Release_Date'] = df['Release_Date'].dt.strftime('%Y-%m-%d')
    
    output_cols = [
        'Period_Date', 'Release_Date', 'Aggregate_Deposits', 'Bank_Credit', 'C/D_differential'
    ]
    final_df = df[output_cols].sort_values('Period_Date', ascending=False)

    # Save to file
    final_df.to_csv(output_path, index=False)
    print(f"Data processed successfully and saved to: {output_path}")

# Run function on your CSV
process_banking_aggregates('Scheduled Commercial Banks - Select Aggregates.csv', 'Banking_Aggregates_Cleaned.csv')

Data processed successfully and saved to: Banking_Aggregates_Cleaned.csv


In [32]:
import pandas as pd
import holidays
from datetime import timedelta

def process_fx_reserves(file_path, output_path):
    # 1. Load the data (headers begin on row 4)
    df = pd.read_csv(file_path, skiprows=3)
    
    # 2. Extract and rename columns
    df = df.iloc[:, [0, 1]].copy()
    df.columns = ['Period_Date_Raw', 'Total_Reserves_Rupees_Raw']
    
    # 3. Clean and parse dates
    df['Period_Date_Raw'] = df['Period_Date_Raw'].astype(str).str.strip()
    df = df[df['Period_Date_Raw'].str.contains('-', na=False)] 
    
    df['Period_Date'] = pd.to_datetime(df['Period_Date_Raw'], errors='coerce', format='%d-%b-%Y')
    df = df.dropna(subset=['Period_Date']).copy()

    # 4. Clean numerical strings
    df['Total_Reserves_Rupees'] = pd.to_numeric(
        df['Total_Reserves_Rupees_Raw'].astype(str).str.replace(',', ''), 
        errors='coerce'
    )
    df = df.dropna(subset=['Total_Reserves_Rupees']).copy()

    # 5. Calculate 4-Week Change (Chronological order required for accurate diff)
    df = df.sort_values(by='Period_Date', ascending=True)
    # .diff(4) calculates: Current Week Value minus Value from 4 rows (weeks) ago
    df['4_Week_Change_Rupees'] = df['Total_Reserves_Rupees'].diff(4)

    # 6. Calculate Release Date with Holiday Adjustments
    start_year = df['Period_Date'].dt.year.min()
    end_year = df['Period_Date'].dt.year.max()
    tg_holidays = holidays.India(years=list(range(int(start_year), int(end_year) + 1)))

    def get_holiday_adjusted_release(date):
        days_to_next_friday = (4 - date.weekday()) + 7
        target_release_friday = date + timedelta(days=days_to_next_friday)
        
        if target_release_friday in tg_holidays:
            actual_release = target_release_friday - timedelta(days=1)
            if actual_release in tg_holidays:
                actual_release = actual_release - timedelta(days=1)
            return actual_release
            
        return target_release_friday

    df['Release_Date'] = df['Period_Date'].apply(get_holiday_adjusted_release)

    # 7. Format Dates and Finalize Columns (Switching back to descending order)
    df['Period_Date'] = df['Period_Date'].dt.strftime('%Y-%m-%d')
    df['Release_Date'] = df['Release_Date'].dt.strftime('%Y-%m-%d')
    
    final_cols = ['Period_Date', 'Release_Date', 'Total_Reserves_Rupees', '4_Week_Change_Rupees']
    final_df = df[final_cols].sort_values(by='Period_Date', ascending=False)

    # 8. Save output
    final_df.to_csv(output_path, index=False)
    print(f"FX Reserves cleanly processed with 4-week changes and saved to: {output_path}")

# Run the function on your dataset
process_fx_reserves('RBIB Table No. 33 _ Foreign Exchange Reserves - Weekly.csv', 'FX_Reserves_4Wk_Metric.csv')

FX Reserves cleanly processed with 4-week changes and saved to: FX_Reserves_4Wk_Metric.csv


In [33]:
import pandas as pd
import os

# --- CONFIGURATION ---
base_file = 'Monthly_Calendar_2010_2026.csv'
source_files = [
    'Banking_Aggregates_Cleaned.csv',
    'FX_Reserves_4Wk_Metric.csv',
    'RBI_Liquidity_With_Lags.csv',
    'WACR_Spread_Automated.csv',
    'TBill_91D_Spread_Clean.csv'
]

# --- 1. PREPARE BASE DATA ---
df_base = pd.read_csv(base_file)
df_base['Last Day of Month'] = pd.to_datetime(df_base['Last Day of Month'])
df_base = df_base.sort_values('Last Day of Month')

# --- 2. CLEAN & MERGE DATA LOOP ---
for file in source_files:
    df_source = pd.read_csv(file)
    
    # Grab the reference date (col 0) and the metric (last col)
    ref_date_col = df_source.columns[0]
    last_col_name = df_source.columns[-1]
    
    df_source = df_source[[ref_date_col, 'Release_Date', last_col_name]].copy()
    df_source['Release_Date'] = pd.to_datetime(df_source['Release_Date'])
    df_source[ref_date_col] = pd.to_datetime(df_source[ref_date_col])
    
    # Drop rows where the Release_Date itself is missing
    df_source = df_source.dropna(subset=['Release_Date'])
    
    # --- NEW: BULLETPROOF CLEANING BEFORE THE MERGE ---
    # Convert the metric to string for aggressive cleaning
    temp_str = df_source[last_col_name].astype(str)
    
    temp_str = temp_str.str.replace(',', '', regex=False)
    temp_str = temp_str.str.replace(r'[−–—]', '-', regex=True) # Fixes fake minus signs
    temp_str = temp_str.str.replace(r'\s+', '', regex=True)
    temp_str = temp_str.str.replace(r'^\((.*)\)$', r'-\1', regex=True)
    temp_str = temp_str.str.replace(r'[₹$%]', '', regex=True)
    
    # Coerce to numeric (unparseable text becomes NaN)
    df_source[last_col_name] = pd.to_numeric(temp_str, errors='coerce')
    
    # --- THE FALLBACK LOGIC ---
    # Drop any rows where the metric is NaN.
    # This forces the upcoming merge_asof to skip empty release dates 
    # and find the next closest one that actually has a value.
    df_source = df_source.dropna(subset=[last_col_name])
    
    # Sort for the backward merge and tie-break by latest reference date
    df_source = df_source.sort_values(['Release_Date', ref_date_col])
    
    # Rename columns to prevent collisions
    file_prefix = os.path.splitext(file)[0][:10]
    new_release_date_col = f'{file_prefix}_Release_Date'
    df_source = df_source.rename(columns={'Release_Date': new_release_date_col})
    df_source = df_source.drop(columns=[ref_date_col])
    
    # Execute the backward merge
    df_base = pd.merge_asof(
        df_base, 
        df_source, 
        left_on='Last Day of Month', 
        right_on=new_release_date_col, 
        direction='backward'
    )

# --- 3. CLEANUP RELEASE DATES ---
cols_to_drop = [col for col in df_base.columns if 'Release_Date' in col]
df_base = df_base.drop(columns=cols_to_drop)

# --- 4. FINAL OUTPUT & ROUNDING ---
# The metrics are already numeric from step 2, so we can just round and save
df_base = df_base.round(4)
df_base.to_csv('Final_Compiled_Data_Raw.csv', index=False)

print("Process complete. Missing values bypassed, data cleaned, and safely merged.")

Process complete. Missing values bypassed, data cleaned, and safely merged.


/var/folders/q5/pq31bcf10fv9ktv6b14g42zr0000gn/T/ipykernel_98103/3633180948.py:77: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_base = df_base.round(4)


In [34]:
import pandas as pd
import numpy as np

def calculate_winsorized_zscores(input_csv, output_csv):

    # ==========================================
    # 1. LOAD & SORT ASCENDING FOR ROLLING
    # ==========================================
    # Must be ascending (oldest → newest) before rolling. Sorting descending
    # first causes each row's window to include future months — the window for
    # March would pull in April and May instead of January and February,
    # corrupting every historical z-score. Only the most recent row was ever
    # correct under the old sort. We flip to descending after rolling for output.
    df = pd.read_csv(input_csv)
    df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values('Last Day of Month', ascending=True).reset_index(drop=True)

    # ==========================================
    # 2. ROBUST Z-SCORE WITH WINSORIZATION
    # ==========================================
    def calc_robust_z_winsorized(window_slice):
        current_val = window_slice[-1]  # last = most recent (ascending sort)

        if np.isnan(current_val):
            return np.nan

        valid_vals = window_slice[~np.isnan(window_slice)]
        if len(valid_vals) == 0:
            return np.nan

        med = np.median(valid_vals)
        mad = np.median(np.abs(valid_vals - med))

        if mad == 0:
            z = 0.0 if (current_val - med) == 0 else np.nan
        else:
            z = (current_val - med) / (1.4826 * mad)

        if not np.isnan(z):
            z = np.clip(z, -4.0, 4.0)

        return z

    # ==========================================
    # 3. APPLY ROLLING Z-SCORE TO EACH COLUMN
    # ==========================================
    value_cols = [
        'C/D_differential',
        '4_Week_Change_Rupees',
        'Net Injection (+)/ Absorption (-) (1+3+5+7+10-2-4-6-9)',
        'WACR_Repo_Spread',
        'TBill_Repo_Spread'
    ]

    z_df = pd.DataFrame()
    z_df['Last Day of Month'] = df['Last Day of Month'].dt.strftime('%Y-%m-%d')

    for col in value_cols:
        z_df[f'{col}_Z'] = (
            df[col]
            .rolling(window=36, min_periods=18)
            .apply(calc_robust_z_winsorized, raw=True)
        )
    
    z_df['C/D_differential_Z'] = z_df['C/D_differential_Z']
    z_df['4_Week_Change_Rupees_Z'] = z_df['4_Week_Change_Rupees_Z'].mul(-1)
    z_df['Net Injection (+)/ Absorption (-) (1+3+5+7+10-2-4-6-9)_Z'] = z_df['Net Injection (+)/ Absorption (-) (1+3+5+7+10-2-4-6-9)_Z']

    # ==========================================
    # 4. SORT DESCENDING (newest → oldest) & EXPORT
    # ==========================================
    z_df = z_df.sort_values('Last Day of Month', ascending=False).reset_index(drop=True)

    z_df.to_csv(output_csv, index=False)
    print(f"Success! Winsorized Z-score file generated and saved to: {output_csv}")


calculate_winsorized_zscores('Final_Compiled_Data_Raw.csv', 'Final_Compiled_Data_Z_Scores.csv')

Success! Winsorized Z-score file generated and saved to: Final_Compiled_Data_Z_Scores.csv


In [35]:
import pandas as pd

def apply_tiered_color_formatting(input_csv, output_xlsx):
    # 1. Load the computed Z-scores CSV
    df = pd.read_csv(input_csv)
    
    # Identify target columns to apply styling (exclude the date column)
    z_cols = [col for col in df.columns if col != 'Last Day of Month']

    # 2. Define multi-tier styling logic
    def format_outliers_tiered(series):
        styles = []
        for val in series:
            if pd.isna(val):
                styles.append('')
            
            # --- POSITIVE DEVIATIONS (RED) ---
            elif val >= 2.0:
                # Strong Outlier: Darker soft red fill, dark text, bold
                styles.append('background-color: #ff9999; color: #660000; font-weight: bold;')
            elif val >= 1.0:
                # Mild Outlier: Lighter soft red fill, regular text
                styles.append('background-color: #ffe6e6; color: #990000;')
                
            # --- NEGATIVE DEVIATIONS (GREEN) ---
            elif val <= -2.0:
                # Strong Outlier: Darker soft green fill, dark text, bold
                styles.append('background-color: #99ff99; color: #004d00; font-weight: bold;')
            elif val <= -1.0:
                # Mild Outlier: Lighter soft green fill, regular text
                styles.append('background-color: #e6ffe6; color: #006600;')
                
            # --- NORMAL RANGE ---
            else:
                styles.append('')
        return styles

    # 3. Use universal .apply() for full cross-version compatibility
    styled_df = df.style.apply(format_outliers_tiered, subset=z_cols, axis=0)

    # 4. Format cells to exactly 4 decimal places
    styled_df = styled_df.format({col: "{:.4f}" for col in z_cols})

    # 5. Save out to Excel file
    styled_df.to_excel(output_xlsx, index=False, engine='openpyxl')
    print(f"Tiered Excel report successfully generated and saved to: {output_xlsx}")

# Run the standalone script on your existing Z-scores CSV
apply_tiered_color_formatting('Final_Compiled_Data_Z_Scores.csv', 'Final_Compiled_Data_Z_Scores_Highlighted.xlsx')

Tiered Excel report successfully generated and saved to: Final_Compiled_Data_Z_Scores_Highlighted.xlsx


In [36]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================================
# 1. STANDARDIZED CALIBRATION (PRESERVED EXACTLY AS YOUR CURRENT CODE)
# =========================================================================
def map_to_5_point_scale(z_score):
    """
    Maps continuous Z-scores onto the 5-point scale: [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0].
    """
    if pd.isna(z_score): 
        return 0.0
    if z_score >= 1.4: 
        return 1.0     # Strongly Stressed / High Vulnerability
    elif z_score >= 1.0: 
        return 0.75     # Moderately Stressed
    elif z_score >= 0.55: 
        return 0.5     # Moderately Stressed
    elif z_score >= 0.25: 
        return 0.25     # Moderately Stressed
    elif z_score <= -1.4: 
        return -1.0    # Strongly Favorable / High Cushion
    elif z_score <= -1.0: 
        return -0.75    # Strongly Favorable / High Cushion
    elif z_score <= -0.55: 
        return -0.5    # Strongly Favorable / High Cushion
    elif z_score <= -0.25: 
        return -0.25    # Moderately Favorable
    else: 
        return 0.0     # Neutral Anchor     

def classify_net_to_7_point_regime(net_score):
    """
    Maps net scores to the true 7-Point Regime Scale: [-3, -2, -1, 0, 1, 2, 3]
    Stabilized with your original thresholds (0.80 / 0.50 / 0.15).
    """
    abs_score = abs(net_score)
    sign = np.sign(net_score)
    
    if abs_score >= 0.8:
        regime = 3.0    # Extreme Liquidity Crisis / Surge
    elif abs_score >= 0.50:
        regime = 2.0    # Deficit / Surplus Acceleration
    elif abs_score >= 0.15:
        regime = 1.0    # Mild Orderly Tightening / Easing
    else:
        regime = 0.0    # Balanced Neutral Anchor
        
    return regime * sign

# =========================================================================
# 2. FALSE-ALARM PROOF TIMING ENGINE (DUAL-PATH ASYMMETRIC FILTER)
# =========================================================================
def apply_validated_asymmetric_filter(regimes, crisis_flags):
    """
    Advanced Chronological Timing Engine.
    - Path A (Fast-Attack): Triggers instantly ONLY if a maximum tail shock 
      is explicitly authenticated by a multi-variable structural blockage flag.
    - Path B (Median Fallback): Subjects standard or unvalidated spikes 
      to the strict 2-out-of-3 chronological majority filter to kill false alarms.
    - Hysteresis Layer (Slow-Decay): Forces a controlled step-down floor 
      when exiting an authenticated crisis state to protect against whipsaws.
    """
    confirmed = []
    current_confirmed = 0.0  # Default neutral anchor
    
    for i in range(len(regimes)):
        flash = regimes[i]
        is_validated_crisis = crisis_flags[i]
        
        if i < 2:
            current_confirmed = flash if pd.notna(flash) else 0.0
            confirmed.append(current_confirmed)
            continue
            
        # 1. AUTHENTICATED FAST-ATTACK GATE
        if (flash == 3.0 or flash == -3.0) and is_validated_crisis:
            current_confirmed = flash
            confirmed.append(current_confirmed)
            continue
            
        # 2. ROBUST MEDIAN FALLBACK GATE (Standard 2-of-3 rule)
        window = [regimes[i], regimes[i-1], regimes[i-2]]
        window = [v for v in window if pd.notna(v)]
        
        if len(window) == 0:
            confirmed.append(current_confirmed)
            continue
            
        counts = pd.Series(window).value_counts()
        highest_frequency = counts.iloc[0]
        most_frequent_value = counts.index[0]
        
        if highest_frequency >= 2:
            proposed_state = most_frequent_value
        else:
            proposed_state = float(np.median(window))
            
        # 3. CONTROLLED SLOW-DECAY HYSTERESIS
        if current_confirmed == 3.0 and proposed_state < 1.0:
            current_confirmed = 1.0  # Orderly step-down safety floor
        elif current_confirmed == -3.0 and proposed_state > -1.0:
            current_confirmed = -1.0
        else:
            current_confirmed = proposed_state
            
        confirmed.append(current_confirmed)
        
    return confirmed

# =========================================================================
# 3. SYSTEMIC LIQUIDITY BLOCKAGE ENGINE & PROCESSING PIPELINE
# =========================================================================
def process_liquidity_pillar(input_excel_path, output_excel_path):
    # 1. Load data natively from Excel
    df = pd.read_excel(input_excel_path)
    
    # Force chronological alignment (Oldest history to Newest) for lookback looping
    df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values('Last Day of Month').reset_index(drop=True)
    
    results = []
    
    # Target Liquidity Weighting Matrix (Sum to 1.0) - Exact match to your notebook
    target_weights = {
        'wacr': 0.25,     # WACR vs Repo Spread (Price Anchor)
        'tbill': 0.15,    # TBill vs Repo Spread (Collateral Velocity)
        'net_inj': 0.30,  # Central Bank Liquidity Operations (Volume Anchor)
        'cd_diff': 0.15,  # Credit minus Deposit Growth Differential (Bank Balance Sheet)
        'cic': 0.15       # Currency in Circulation 4-Week Change (Leakage)
    }
    
    # Exact column labels inside your sheet
    col_inj = 'Net Injection (+)/ Absorption (-) (1+3+5+7+10-2-4-6-9)_Z'
    
    for idx, row in df.iterrows():
        wacr_z = row['WACR_Repo_Spread_Z']
        tbill_z = row['TBill_Repo_Spread_Z']
        cd_z = row['C/D_differential_Z']
        cic_z = row['4_Week_Change_Rupees_Z']
        inj_z = row[col_inj]
        
        # Map variables to the 5-point scale
        wacr_f = map_to_5_point_scale(wacr_z) if pd.notna(wacr_z) else None
        tbill_f = map_to_5_point_scale(tbill_z) if pd.notna(tbill_z) else None
        cd_f = map_to_5_point_scale(cd_z) if pd.notna(cd_z) else None
        cic_f = map_to_5_point_scale(cic_z) if pd.notna(cic_z) else None
        
        # Central Bank absorpotion map as loose (banking system is too flush with cash)
        net_inj_f = map_to_5_point_scale(inj_z) if pd.notna(inj_z) else None
        
        weighted_sum = 0.0
        total_active_weight = 0.0
        
        if wacr_f is not None: weighted_sum += wacr_f * target_weights['wacr']; total_active_weight += target_weights['wacr']
        if tbill_f is not None: weighted_sum += tbill_f * target_weights['tbill']; total_active_weight += target_weights['tbill']
        if cd_f is not None: weighted_sum += cd_f * target_weights['cd_diff']; total_active_weight += target_weights['cd_diff']
        if cic_f is not None: weighted_sum += cic_f * target_weights['cic']; total_active_weight += target_weights['cic']
        if net_inj_f is not None: weighted_sum += net_inj_f * target_weights['net_inj']; total_active_weight += target_weights['net_inj']
            
        if total_active_weight == 0.0:
            results.append({'Base_Net_Score': np.nan, 'Adjusted_Net_Score': np.nan, 'Flash_Regime': np.nan, 'Crisis_Flag': False})
            continue
            
        base_net_score = weighted_sum / total_active_weight
        
        # 2. SYSTEMIC TRANSMISSION FAILURE / BLOCKAGE MODIFIER (Exact match to notebook)
        if wacr_f is not None and net_inj_f is not None and (wacr_f * net_inj_f < 0):
            divergence_magnitude = abs(wacr_f - net_inj_f)
            if wacr_f > 0 and net_inj_f < 0:
                # Credit Blockage Trap: RBI is injecting cash but interbank rates remain spiked
                dislocation_modifier = 0.15 * (divergence_magnitude / 2.0)
            else:
                # Flush Saturation: RBI is trying to absorb cash but overnight rates stay deeply depressed
                dislocation_modifier = -0.15 * (divergence_magnitude / 2.0)
            adjusted_net_score = base_net_score + dislocation_modifier
        else:
            adjusted_net_score = base_net_score
            
        # Hard mathematical boundary clamp keeping scores strictly within [-1.0, 1.0]
        final_net_score = max(-1.0, min(1.0, adjusted_net_score))
        flash_regime = classify_net_to_7_point_regime(final_net_score)
        
        # 3. TIE-IN TO ASYMMETRIC FILTER
        # Authenticate a systemic shock if both WACR and CB Injections hit their extreme 
        # identical limit concurrently, confirming structural plumbing failure.
        if wacr_f is not None and net_inj_f is not None:
            structural_crisis = (wacr_f == 1.0 and net_inj_f == 1.0) or (wacr_f == -1.0 and net_inj_f == -1.0)
        else:
            structural_crisis = False
        
        results.append({
            'Base_Net_Score': base_net_score,
            'Adjusted_Net_Score': adjusted_net_score,
            'Flash_Regime': flash_regime,
            'Crisis_Flag': structural_crisis
        })

    # Combine calculations back into the dataframe
    res_df = pd.DataFrame(results)
    output_df = pd.concat([df, res_df], axis=1)
    
    # 4. RUN CHRONOLOGICAL LOOKBACK FILTER
    output_df['Liquidity_Confirmed_Regime'] = apply_validated_asymmetric_filter(
        output_df['Flash_Regime'].tolist(),
        output_df['Crisis_Flag'].tolist()
    )
    
    # Drop the intermediate flash and tracking columns to leave only the finalized confirmed output
    output_df = output_df.drop(columns=['Flash_Regime', 'Crisis_Flag'])
    
    # PRESENTATION FLIP: Reverse sort table so 2026 sits cleanly at the top of your sheet
    output_df = output_df.sort_values('Last Day of Month', ascending=False).reset_index(drop=True)
    
    # Save base dataframe to Excel
    output_df.to_excel(output_excel_path, index=False)
    
    # 5. SHEET STYLING & PROFESSIONAL POLISHING (Openpyxl Engine - No Row Color Scaling)
    wb = openpyxl.load_workbook(output_excel_path)
    ws = wb.active
    ws.title = "Liquidity Model Analysis"
    
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
    )
    
    # Format Headers
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        
    # Format Data Rows
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.border = thin_border
            if cell.column == 1:
                if isinstance(cell.value, (pd.Timestamp, np.datetime64)) or (isinstance(cell.value, str) and '-' in str(cell.value)):
                    cell.number_format = 'yyyy-mm-dd'
                cell.alignment = Alignment(horizontal="center") 
            else:
                cell.alignment = Alignment(horizontal="right")   
                if cell.value is not None and isinstance(cell.value, (int, float)):
                    cell.number_format = '0.00'                  

    # Auto-adjust column widths dynamically to prevent cellular text clipping
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = openpyxl.utils.get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)
        
    # Freeze header panel and dates column securely in place
    ws.freeze_panes = 'B2'
    
    wb.save(output_excel_path)
    print(f"Process complete! Output successfully saved to: {output_excel_path}")

# Execute using your native uploaded spreadsheet
if __name__ == "__main__":
    process_liquidity_pillar("Final_Compiled_Data_Z_Scores_Highlighted.xlsx", "Liquidity_Pillar_Model_Outputs.xlsx")

Process complete! Output successfully saved to: Liquidity_Pillar_Model_Outputs.xlsx
